# 07 — Cross-Representation Integration
Integrates clustering results from NMF and SVD across the three molecular
representation spaces (fingerprint, image, graph) to identify a final set of
candidate molecules consistently co-localised with the reference compound.

**Pipeline:**
1. Intersect the three NMF reference clusters (fingerprint ∩ image ∩ graph)
2. Intersect the three SVD reference clusters (fingerprint ∩ image ∩ graph)
3. Unite NMF and SVD intersections → final candidate indices
4. Extract molecule metadata from SDF and save CSV + SDF

**Input:** `.npy` files produced by notebooks `05_nmf_clustering.ipynb`
and `06_svd_clustering.ipynb`.

**Output:**
- `indices_final.npy` — final candidate indices
- `candidate_molecules.csv` — Index + Compound_ID for each candidate
- `candidate_molecules.sdf` — SDF file of candidate molecules

## Configuration

In [ ]:
# NMF cluster indices (one per representation) 
PATH_NMF_FP    = "data/nmf_results/fingerprint/indices_cluster_NMF.npy"
PATH_NMF_IMG   = "data/nmf_results/image/indices_cluster_NMF.npy"
PATH_NMF_GRAPH = "data/nmf_results/graph/indices_cluster_NMF.npy"

# SVD cluster indices (one per representation) 
PATH_SVD_FP    = "data/svd_results/fingerprint/indices_cluster_SVD.npy"
PATH_SVD_IMG   = "data/svd_results/image/indices_cluster_SVD.npy"
PATH_SVD_GRAPH = "data/svd_results/graph/indices_cluster_SVD.npy"

# Input SDF (full dataset) 
INPUT_SDF = "data/molecules.sdf"

# Output 
OUTPUT_DIR        = "data/integration/"
OUTPUT_NPY        = "data/integration/indices_final.npy"
OUTPUT_CSV        = "data/integration/candidate_molecules.csv"
OUTPUT_SDF        = "data/integration/candidate_molecules.sdf"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Dependencies

In [ ]:
# pip install numpy pandas rdkit matplotlib matplotlib-venn

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from rdkit import Chem

## 2. Load cluster indices
Load the indices of molecules assigned to the reference cluster
for each representation and each dimensionality reduction method.

In [ ]:
# NMF
nmf_fp    = set(np.load(PATH_NMF_FP))
nmf_img   = set(np.load(PATH_NMF_IMG))
nmf_graph = set(np.load(PATH_NMF_GRAPH))

# SVD
svd_fp    = set(np.load(PATH_SVD_FP))
svd_img   = set(np.load(PATH_SVD_IMG))
svd_graph = set(np.load(PATH_SVD_GRAPH))

print("NMF cluster sizes:")
print(f"  Fingerprint : {len(nmf_fp)}")
print(f"  Image       : {len(nmf_img)}")
print(f"  Graph       : {len(nmf_graph)}")

print("\nSVD cluster sizes:")
print(f"  Fingerprint : {len(svd_fp)}")
print(f"  Image       : {len(svd_img)}")
print(f"  Graph       : {len(svd_graph)}")

## 3. Cross-representation intersection
Retain only molecules that appear in the reference cluster across
**all three representations** for each dimensionality reduction method.

In [ ]:
indices_overlap_NMF = sorted(nmf_fp & nmf_img & nmf_graph)
indices_overlap_SVD = sorted(svd_fp & svd_img & svd_graph)

print(f"NMF intersection (FP ∩ IMG ∩ GRAPH) : {len(indices_overlap_NMF)} molecules")
print(f"SVD intersection (FP ∩ IMG ∩ GRAPH) : {len(indices_overlap_SVD)} molecules")
print(f"\nNMF indices: {indices_overlap_NMF}")
print(f"SVD indices: {indices_overlap_SVD}")

## 4. Venn diagrams
Visualise the overlap between the three representation spaces
for NMF and SVD separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (fp, img, graph, title) in zip(axes, [
    (nmf_fp, nmf_img, nmf_graph, 'NMF — cross-representation overlap'),
    (svd_fp, svd_img, svd_graph, 'SVD — cross-representation overlap'),
]):
    plt.sca(ax)
    venn3([fp, img, graph], set_labels=('Fingerprint', 'Image', 'Graph'))
    ax.set_title(title, fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'venn_overlap.tif'),
            dpi=300, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
plt.show()

## 5. Final union
Unite the NMF and SVD intersections to obtain the final set of candidate molecules.
A molecule is retained if it appears in the NMF intersection, the SVD intersection, or both.

In [ ]:
indices_final = np.array(sorted(set(indices_overlap_NMF) | set(indices_overlap_SVD)))

print(f"NMF intersection : {len(indices_overlap_NMF)} molecules")
print(f"SVD intersection : {len(indices_overlap_SVD)} molecules")
print(f"Final union      : {len(indices_final)} unique molecules")
print(f"\nFinal indices: {indices_final}")

## 6. Extract molecule metadata from SDF

In [ ]:
supplier = Chem.SDMolSupplier(INPUT_SDF)
print(f"Total molecules in SDF: {len(supplier)}")

# Properties to try in order for Compound_ID
ID_PROPS = ["PCAT_CMPD_ID", "PUBCHEM_CID", "PUBCHEM_COMPOUND_CID",
            "CID", "CHEMBL_ID", "MERGED_SDF_INDEX", "ID", "Name"]

# Set degli indici individuati da ciascuna tecnica di riduzione, per marcare NMF/SVD
set_nmf = set(int(i) for i in indices_overlap_NMF)
set_svd = set(int(i) for i in indices_overlap_SVD)

data = []
for i in indices_final:
    i = int(i)
    if i >= len(supplier):
        print(f"Index out of range: {i}")
        continue
    mol = supplier[i]
    if mol is None:
        print(f"None molecule at index: {i}")
        continue

    entry = {"Index": i}

    # Compound_ID: use first available property from ID_PROPS
    compound_id = f"mol_{i}"  # fallback
    for prop in ID_PROPS:
        if mol.HasProp(prop):
            compound_id = mol.GetProp(prop)
            break
    entry["Compound_ID"] = compound_id

    # Tecnica/e di riduzione dimensionale con cui la molecola e' stata individuata
    entry["NMF"] = "Si" if i in set_nmf else "No"
    entry["SVD"] = "Si" if i in set_svd else "No"

    # Additional identifiers if available
    for prop in ["PUBCHEM_IUPAC_NAME", "PUBCHEM_IUPAC_INCHIKEY",
                 "PUBCHEM_OPENEYE_ISO_SMILES"]:
        if mol.HasProp(prop):
            entry[prop] = mol.GetProp(prop)

    data.append(entry)

# Print available properties from first valid molecule (useful for debugging)
first_mol = next((supplier[int(i)] for i in indices_final
                  if supplier[int(i)] is not None), None)
if first_mol:
    print(f"\nAvailable properties in SDF: {list(first_mol.GetPropsAsDict().keys())}")

print(f"\nMolecules extracted: {len(data)}")
print(f"Molecules found by BOTH NMF and SVD: {sum(1 for e in data if e['NMF'] == 'Si' and e['SVD'] == 'Si')}")

## 7. Save CSV

In [ ]:
df_final = pd.DataFrame(data).sort_values("Compound_ID").reset_index(drop=True)
df_final.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
display(df_final)

## 8. Save final indices (.npy)

In [ ]:
np.save(OUTPUT_NPY, indices_final)
print(f"Saved: {OUTPUT_NPY}  ({len(indices_final)} indices)")

## 9. Save SDF of candidate molecules

In [ ]:
writer = Chem.SDWriter(OUTPUT_SDF)
count  = 0

for i in indices_final:
    i = int(i)
    if i >= len(supplier):
        print(f"Index out of range: {i}")
        continue
    mol = supplier[i]
    if mol is None:
        print(f"None molecule at index: {i}")
        continue

    # Set molecule name using first available ID property
    for prop in ID_PROPS:
        if mol.HasProp(prop):
            mol.SetProp("_Name", mol.GetProp(prop))
            break
    else:
        mol.SetProp("_Name", f"mol_{i}")

    writer.write(mol)
    count += 1

writer.close()
print(f"Saved: {OUTPUT_SDF}  ({count} molecules)")